In [2]:
import pandas as pd
import numpy as np

from skeLCS.eLCS import eLCS

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

data_raw = pd.read_csv("../data/weatherAUS.csv")

print("Original shape:", data_raw.shape)
print(data_raw.head())


Original shape: (145460, 23)
         Date Location  MinTemp  MaxTemp  Rainfall  Evaporation  Sunshine  \
0  2008-12-01   Albury     13.4     22.9       0.6          NaN       NaN   
1  2008-12-02   Albury      7.4     25.1       0.0          NaN       NaN   
2  2008-12-03   Albury     12.9     25.7       0.0          NaN       NaN   
3  2008-12-04   Albury      9.2     28.0       0.0          NaN       NaN   
4  2008-12-05   Albury     17.5     32.3       1.0          NaN       NaN   

  WindGustDir  WindGustSpeed WindDir9am  ... Humidity9am  Humidity3pm  \
0           W           44.0          W  ...        71.0         22.0   
1         WNW           44.0        NNW  ...        44.0         25.0   
2         WSW           46.0          W  ...        38.0         30.0   
3          NE           24.0         SE  ...        45.0         16.0   
4           W           41.0        ENE  ...        82.0         33.0   

   Pressure9am  Pressure3pm  Cloud9am  Cloud3pm  Temp9am  Temp3pm  Ra

In [3]:
# remove rows where the target is missing
data_raw = data_raw.dropna(
    subset=["RainTomorrow"]
).copy()

# convert target to numeric
data_raw["RainTomorrow"] = data_raw["RainTomorrow"].map({
    "No": 0,
    "Yes": 1
})

# Date is removed for the raw baseline rather than feature engineered
data_raw = data_raw.drop(columns=["Date"])

print("Shape after removing missing target:", data_raw.shape)

print("\nTarget distribution:")
print(data_raw["RainTomorrow"].value_counts())

print("\nMissing feature values:")
print(data_raw.isna().sum().sort_values(ascending=False))


Shape after removing missing target: (142193, 22)

Target distribution:
RainTomorrow
0    110316
1     31877
Name: count, dtype: int64

Missing feature values:
Sunshine         67816
Evaporation      60843
Cloud3pm         57094
Cloud9am         53657
Pressure9am      14014
Pressure3pm      13981
WindDir9am       10013
WindGustDir       9330
WindGustSpeed     9270
WindDir3pm        3778
Humidity3pm       3610
Temp3pm           2726
WindSpeed3pm      2630
Humidity9am       1774
RainToday         1406
Rainfall          1406
WindSpeed9am      1348
Temp9am            904
MinTemp            637
MaxTemp            322
Location             0
RainTomorrow         0
dtype: int64


In [4]:
classlabel = "RainTomorrow"

x_raw = data_raw.drop(columns=[classlabel])
y_raw = data_raw[classlabel]

x_train, x_test, y_train, y_test = train_test_split(
    x_raw,
    y_raw,
    test_size=0.20,
    random_state=42,
    stratify=y_raw
)

print("Training:", x_train.shape)
print("Testing:", x_test.shape)

print("\nTraining target:")
print(y_train.value_counts())

print("\nTesting target:")
print(y_test.value_counts())


Training: (113754, 21)
Testing: (28439, 21)

Training target:
RainTomorrow
0    88252
1    25502
Name: count, dtype: int64

Testing target:
RainTomorrow
0    22064
1     6375
Name: count, dtype: int64


In [5]:
# categorical values must be numeric for eLCS
categorical_cols = x_train.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

print("Categorical columns:")
print(categorical_cols)

encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

x_train = x_train.copy()
x_test = x_test.copy()

x_train[categorical_cols] = encoder.fit_transform(
    x_train[categorical_cols]
)

x_test[categorical_cols] = encoder.transform(
    x_test[categorical_cols]
)


Categorical columns:
['Location', 'WindGustDir', 'WindDir9am', 'WindDir3pm', 'RainToday']


In [6]:
# tell eLCS which encoded columns are discrete categories
discrete_attributes = np.array([
    x_train.columns.get_loc(col)
    for col in categorical_cols
])

print("Discrete attribute indices:")
print(discrete_attributes)

# professor's eLCS requires numeric numpy arrays
x_train_np = x_train.to_numpy(dtype=float)
x_test_np = x_test.to_numpy(dtype=float)

y_train_np = y_train.to_numpy(dtype=float)
y_test_np = y_test.to_numpy(dtype=float)

print("\nTraining shape:", x_train_np.shape)
print("Testing shape:", x_test_np.shape)
print("Features:", x_train_np.shape[1])

print("\nMissing training feature values:",
      np.isnan(x_train_np).sum())

print("Missing testing feature values:",
      np.isnan(x_test_np).sum())


Discrete attribute indices:
[ 0  6  8  9 20]

Training shape: (113754, 21)
Testing shape: (28439, 21)
Features: 21

Missing training feature values: 253678
Missing testing feature values: 62881


In [7]:
# small raw eLCS test run before the full experiment
x_sample, _, y_sample, _ = train_test_split(
    x_train_np,
    y_train_np,
    train_size=25000,
    random_state=42,
    stratify=y_train_np
)

raw_test_model = eLCS(
    learning_iterations=5000,
    N=1000,
    discrete_attribute_limit="d",
    specified_attributes=discrete_attributes,
    random_state=42
)

raw_test_model.fit(
    x_sample,
    y_sample
)

print("Raw eLCS test training complete")


Raw eLCS test training complete


In [8]:
raw_test_predictions = raw_test_model.predict(
    x_test_np
)

print(
    "Accuracy:",
    accuracy_score(
        y_test_np,
        raw_test_predictions
    )
)

print(
    "Balanced Accuracy:",
    balanced_accuracy_score(
        y_test_np,
        raw_test_predictions
    )
)

print("\nClassification Report:")
print(
    classification_report(
        y_test_np,
        raw_test_predictions
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test_np,
        raw_test_predictions
    )
)


Accuracy: 0.7966876472449805
Balanced Accuracy: 0.5524771964623413

Classification Report:
              precision    recall  f1-score   support

         0.0       0.79      1.00      0.88     22064
         1.0       0.87      0.11      0.19      6375

    accuracy                           0.80     28439
   macro avg       0.83      0.55      0.54     28439
weighted avg       0.81      0.80      0.73     28439


Confusion Matrix:
[[21957   107]
 [ 5675   700]]


In [12]:
# full raw eLCS baseline
# learning iterations match the number of training instances
raw_elcs = eLCS(
    learning_iterations=len(x_train_np),
    N=7000,
    discrete_attribute_limit="d",
    specified_attributes=discrete_attributes,
    random_state=42
)

raw_elcs.fit(
    x_train_np,
    y_train_np
)

print("Raw eLCS training complete")


Raw eLCS training complete


In [13]:
raw_predictions = raw_elcs.predict(
    x_test_np
)

print(
    "Accuracy:",
    accuracy_score(
        y_test_np,
        raw_predictions
    )
)

print(
    "Balanced Accuracy:",
    balanced_accuracy_score(
        y_test_np,
        raw_predictions
    )
)

print("\nClassification Report:")
print(
    classification_report(
        y_test_np,
        raw_predictions
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test_np,
        raw_predictions
    )
)


Accuracy: 0.8286859594219206
Balanced Accuracy: 0.6356729905160033

Classification Report:
              precision    recall  f1-score   support

         0.0       0.83      0.99      0.90     22064
         1.0       0.85      0.29      0.43      6375

    accuracy                           0.83     28439
   macro avg       0.84      0.64      0.66     28439
weighted avg       0.83      0.83      0.79     28439


Confusion Matrix:
[[21745   319]
 [ 4553  1822]]


In [14]:
import os

os.makedirs("../results", exist_ok=True)

raw_results = pd.DataFrame({
    "actual": y_test_np,
    "prediction": raw_predictions
})

raw_results.to_csv(
    "../results/raw_elcs_predictions.csv",
    index=False
)

raw_elcs.export_iteration_tracking_data(
    "../results/raw_elcs_iterations.csv"
)

raw_elcs.export_final_rule_population(
    headerNames=x_train.columns.to_numpy(),
    className="RainTomorrow",
    filename="../results/raw_elcs_rules.csv"
)

print("Raw eLCS results saved")


Raw eLCS results saved
